In [0]:
%run "./0.0 Utils"

In [0]:
# =========================================================
# Parameter widgets
# =========================================================

dbutils.widgets.removeAll()

dbutils.widgets.text("BRONZE_SUBPATH", "bronze", "Bronze Subpath")
dbutils.widgets.text("CATALOG_NAME", "workspace", "UC Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "default", "UC Schema Name")
dbutils.widgets.text("TABLE_NAME", "fct_activity", "UC Table Name")

bronze_subpath = dbutils.widgets.get("BRONZE_SUBPATH").strip().strip("/")
catalog_name   = dbutils.widgets.get("CATALOG_NAME").strip()
schema_name    = dbutils.widgets.get("SCHEMA_NAME").strip()
table_name     = dbutils.widgets.get("TABLE_NAME").strip()

In [0]:
# =========================================================
# DESCRIPTIVE BLOCK — BRONZE TABLE SCHEMAS
# =========================================================
import os
def read_csv(path: str):
    """Read a CSV with common options used across Bronze inputs."""
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "false")
        .option("escape", "\"")
        .load(path)
    )

# ---------------------------------------------------------
# Print schema and row count for each Bronze table
# ---------------------------------------------------------

directory_main_bronze = f"{directory_notebook}/{bronze_subpath}"

bronze_tables = ["events.csv","battles.csv","characters.csv","planets.csv","starships.csv"]

for file in bronze_tables:
    path = f"{directory_main_bronze}/{file}"
    print(f"\n=== Schema for {file} ===")
    df = read_csv(path)
    df.printSchema()
    try:
        total_rows = df.count()
    except Exception as e:
        total_rows = f"count() failed: {e}"
    print(f"Total rows: {total_rows}")
    print("=" * 60)


In [0]:
from pyspark.sql import functions as F, types as T
import secrets

# =========================================================
# Path setup
# =========================================================

bronze = f"{directory_notebook}/{bronze_subpath}"

# Unity Catalog table (fully qualified)
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

# =========================================================
# Execution metadata
# =========================================================

_exec_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]

_exec_id = secrets.token_hex(4)  

EXEC_TS_LIT = F.lit(_exec_ts)
EXEC_ID_LIT = F.lit(_exec_id)

# =========================================================
# Read Bronze CSV files
# =========================================================

def read_csv(path: str):
    """Generic CSV reader with common options."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "false")
        .option("escape", "\"")
        .load(path)
    )

events     = read_csv(f"{bronze}/events.csv")
battles    = read_csv(f"{bronze}/battles.csv")
characters = read_csv(f"{bronze}/characters.csv")
planets    = read_csv(f"{bronze}/planets.csv")
starships  = read_csv(f"{bronze}/starships.csv")

# =========================================================
# Basic transformations (no enrichment)
# =========================================================

# EVENTS 
events_silver = (
    events
    .withColumnRenamed("id", "event_id")
    .withColumnRenamed("event_name", "name")
    .drop("date")
    .withColumn("location", F.initcap(F.trim("location")))
)

# BATTLES 
battles_silver = (
    battles
    .withColumnRenamed("id", "battle_id")
    .withColumnRenamed("name", "battle_name")
    .drop("date")
    .withColumn("location", F.initcap(F.trim("location")))
)

# CHARACTERS only in the top-3 species

common_species = spark.read.parquet(f"file:{os.getcwd()}/sesion_2/common_species")

characters_silver = (
    characters.join(common_species, ["species"], "inner")
    .withColumnRenamed("id", "character_id")
    .withColumn("name", F.initcap(F.trim("name")))
    .withColumn("gender", F.initcap(F.col("gender")))
    .dropDuplicates(["character_id"])
)

# PLANETS 
planets_silver = (
    planets
    .withColumnRenamed("id", "planet_id")
    .withColumn("name", F.initcap(F.trim("name")))
    .withColumn("climate", F.initcap(F.col("climate")))
    .withColumn("terrain", F.initcap(F.col("terrain")))
    .withColumn("population", F.col("population").cast("long"))
    .dropDuplicates(["planet_id"])
)

# STARSHIPS 
starships_silver = (
    starships
    .withColumnRenamed("id", "ship_id")
    .withColumn("name", F.initcap(F.col("name")))
    .withColumn("manufacturer", F.initcap(F.col("manufacturer")))
    .withColumn("starship_class", F.initcap(F.col("starship_class")))
    .dropDuplicates(["ship_id"])
)

# =========================================================
# Simple union: Events + Battles
# =========================================================

events_fct = (
    events_silver
    .withColumn("activity_type", F.lit("event"))
    .withColumnRenamed("name", "activity_name")
    .select("activity_type", "activity_name", "location", "description")
)

battles_fct = (
    battles_silver
    .withColumn("activity_type", F.lit("battle"))
    .withColumnRenamed("battle_name", "activity_name")
    .select("activity_type", "activity_name", "location", F.col("result").alias("description"))
)

fct_activity = events_fct.unionByName(battles_fct, allowMissingColumns=True)

# =========================================================
# Write to Unity Catalog Delta table
# =========================================================

(
    fct_activity
    .withColumn("process_run_ts", EXEC_TS_LIT)
    .withColumn("process_run_id", EXEC_ID_LIT)
    .write
    .format("delta")
    .mode("append")               # append each run incrementally
    .option("mergeSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"✓ Successfully written Delta data to Unity Catalog table: {full_table_name}")
print(f"  Process timestamp:  {_exec_ts}")
print(f"  Process run ID:     {_exec_id}")
print(f"  Rows written:       {fct_activity.count()}")


In [0]:
# Query the Unity Catalog table
print(f"Reading from table: {full_table_name}")
display(spark.table(full_table_name))